# 圖書館借閱管理系統

**資料庫管理**・教師示範專題　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/demo/library/library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

**這是一個「做完了」的專題長的樣子**——共同要求 10 條全數達標。（本題**不在**指派清單內：請模仿它的結構與完成度，別抄它的程式。）

> **怎麼使用這本 notebook**
> - **當標竿**：右欄目錄就是一份專題的完整結構；每一節開頭都標了它對應「共同要求」第幾條。
> - **當導讀**：markdown 是講解稿；程式格每段做一件事，前面都有一句「現在要做什麼」。
> - **當零件庫**：交易寫法、競態示範、報表 SQL、Gradio 佈局——結構可以學走，**內容請換成你自己的題目**（報告 Q&A 考的是你講不講得出來）。
>
> 全本可重跑：`執行階段 → 全部執行`，約 1 分鐘（最後的 `launch()` 會開出可操作介面）。
> 本示範把「今天」固定為 `DEMO_TODAY = "2026-11-15"`；資料、業務函數與 UI 共用同一日期，
> 因此不會隨執行當天漂移。UI 可輸入其他 `YYYY-MM-DD` 日期重演情境。

## 共同要求 10 條 ↔ 本 notebook 對照

| # | 要求 | 在哪一節 |
|---|---|---|
| 1 | ≥4 表 3NF＋約束＋schema 圖 | §1–§2 |
| 2 | 固定 seed 合成 ≥10,000 列 | §3 |
| 3 | CRUD＋`?` 傳值＋表單驗證 | §4、§8 |
| 4 | 交易保護＋併發競態示範 | §4、**§5** |
| 5 | 報表 ≥5（window／join／圖） | §6 |
| 6 | 最慢查詢 EXPLAIN＋索引前後計時 | §7 |
| 7 | Gradio ≥3 分頁 | §8 |
| 8 | ≥8 assert（含 2 個應該失敗） | 各節隨做隨測＋§9 總驗收 |
| 9 | AI 使用說明 | §10 |
| 10 | 簡報與 demo 腳本 | §11 |

# §1 情境、需求與 schema 設計（共同要求 1）

## 1.1 情境訪談稿

> 「系圖書室有幾百本**書**，每本可能有**好幾冊**。**讀者**辦證後可以**借書**，
> 一次借 28 天，可以**續借**兩次——但有人在**預約**這本書就不能續。
> 冊數借完了，讀者可以留下預約，**還書時通知排最前面的人**。
> 我們想知道：哪些書熱門？誰逾期了？每月借閱量的趨勢？」

## 1.2 名詞動詞分析（U04 的 SOP）

| 名詞 → 實體 | 動詞 → 關係 | 藏在句子裡的規則 |
|---|---|---|
| 書 book（冊數是屬性） | 讀者**借**書 → loan（M:N＋屬性） | 借期 28 天、續借 ≤2 次 |
| 讀者 member | 讀者**預約**書 → reservation（M:N＋屬性） | 同書同人只能有一筆「等待中」 |
| 借閱 loan（弱實體感：靠 book+member+時間識別，用代理鍵） | 還書**喚醒**預約 | 還書＝結案＋回庫＋通知，**一個交易** |

## 1.3 ER 圖與設計決策

```
 member ──1───借───N── loan ──N───借的是───1── book
 (讀者)               (借閱紀錄)                (書目)
 member ──1───約───N── reservation ──N───約的是───1── book
 (讀者)               (預約，帶狀態)             (書目)
```

三條**設計決策**（報告 Q&A 必問，先寫下來）：
1. **庫存用「快照＋對帳」**：`book.available_copies` 存目前可借冊數（櫃台查詢極頻繁），
   由借／還交易同步維護；§3 末用「總冊數 − 在外未還」對帳驗證（U04 模式⑤）。
2. **「等待中的預約不可重複」用部分唯一索引**：`UNIQUE(book_id, member_id) WHERE status='等待'`——
   取消後可以再約（歷史保留），等待中不准重複。（部分索引在 U07 §2.4 正式拆解；這裡先會用。）
3. **loan 用代理鍵**：同人同書可多次借還，(book, member) 不唯一；`return_date IS NULL` ＝ 在外。

In [ ]:
#@title 📦 §2 建庫：DDL（4 表、PK/FK、NOT NULL／UNIQUE／CHECK／DEFAULT 全用上）
import sqlite3, os
from datetime import date, timedelta
import numpy as np, pandas as pd

DEMO_TODAY = "2026-11-15"

def validate_iso_date(value, field="日期"):
    # 接受嚴格 YYYY-MM-DD；回傳正規化字串，讓 SQL 與 UI 共用。
    text = str(value or "").strip()
    try:
        parsed = date.fromisoformat(text)
    except (TypeError, ValueError):
        raise ValueError(f"{field}請用 YYYY-MM-DD（例如 {DEMO_TODAY}）")
    if parsed.isoformat() != text:
        raise ValueError(f"{field}請用 YYYY-MM-DD（例如 {DEMO_TODAY}）")
    return text

assert validate_iso_date(DEMO_TODAY) == DEMO_TODAY

if os.path.exists("library.db"):
    os.remove("library.db")
lcon = sqlite3.connect("library.db", check_same_thread=False)   # check_same_thread：給稍後的 Gradio 用
lcon.row_factory = sqlite3.Row
lcon.executescript(f"""
PRAGMA foreign_keys = ON;

CREATE TABLE book(
  book_id          INTEGER PRIMARY KEY,
  title            TEXT NOT NULL,
  author           TEXT NOT NULL,
  isbn             TEXT UNIQUE,                          -- 可能缺（舊書），有就不准重複
  category         TEXT NOT NULL CHECK (category IN
                     ('統計','數學','資訊','小說','商管','科普','語言','漫畫')),
  total_copies     INTEGER NOT NULL CHECK (total_copies > 0),
  available_copies INTEGER NOT NULL
                   CHECK (available_copies BETWEEN 0 AND total_copies)   -- 快照欄＋雙邊界
);

CREATE TABLE member(
  member_id INTEGER PRIMARY KEY,
  name      TEXT NOT NULL,
  email     TEXT NOT NULL UNIQUE,
  joined    TEXT NOT NULL DEFAULT ('{DEMO_TODAY}')
);

CREATE TABLE loan(
  loan_id     INTEGER PRIMARY KEY,
  book_id     INTEGER NOT NULL REFERENCES book(book_id),
  member_id   INTEGER NOT NULL REFERENCES member(member_id),
  loan_date   TEXT NOT NULL,
  due_date    TEXT NOT NULL,
  return_date TEXT,                                      -- NULL = 還在外面
  renew_count INTEGER NOT NULL DEFAULT 0 CHECK (renew_count BETWEEN 0 AND 2),
  CHECK (loan_date <= due_date),
  CHECK (return_date IS NULL OR return_date >= loan_date)
);

CREATE TABLE reservation(
  res_id    INTEGER PRIMARY KEY,
  book_id   INTEGER NOT NULL REFERENCES book(book_id),
  member_id INTEGER NOT NULL REFERENCES member(member_id),
  res_time  TEXT NOT NULL DEFAULT ('{DEMO_TODAY} 12:00:00'),
  status    TEXT NOT NULL DEFAULT '等待' CHECK (status IN ('等待','已通知','取消','完成'))
);
-- 設計決策 2：等待中的預約，同書同人只能一筆（部分唯一索引；歷史紀錄不受影響）
CREATE UNIQUE INDEX ux_res_waiting ON reservation(book_id, member_id) WHERE status = '等待';
""")
print(f"library.db 就緒 ✅ —— 模擬今日 {DEMO_TODAY}；4 表、4 條 FK；"
      "PK/FK 之外的約束型別 NOT NULL / UNIQUE / CHECK / DEFAULT 全到齊")
print([r[0] for r in lcon.execute("SELECT name FROM sqlite_master WHERE type='table'")])

In [ ]:
# 約束踩點：每條規則踩一腳，全都要被擋下（這 4 個之後也算進 §9 的「應該失敗」測試）
trials = [
    ("冊數開 0",        "INSERT INTO book(title,author,category,total_copies,available_copies) "
                        "VALUES ('x','y','統計',0,0)"),
    ("可借 > 總冊數",    "INSERT INTO book(title,author,category,total_copies,available_copies) "
                        "VALUES ('x','y','統計',2,3)"),
    ("亂寫類別",        "INSERT INTO book(title,author,category,total_copies,available_copies) "
                        "VALUES ('x','y','八卦',1,1)"),
    ("幽靈讀者借書",     "INSERT INTO loan(book_id,member_id,loan_date,due_date) "
                        "VALUES (1, 99999, '2026-01-01','2026-01-29')"),
]
for label, sql in trials:
    try:
        lcon.execute(sql); print(f"⚠️ {label}：竟然過了？！")
    except sqlite3.IntegrityError as e:
        print(f"✅ {label:10s} 擋下 → {e}")
lcon.rollback()

# §3 合成擬真資料（共同要求 2）：固定 seed、13,000+ 列

**生成假設**（這段就是報告裡的「資料說明」）：

- **使用者是誰**：一個系圖書室——800 種書（1–5 冊）、500 位讀者、約兩學年（2025-09 起）的流通史。
- **行為規律**（U05 的七講究，這裡用了四個）：
  1. **長尾**：少數熱門書貢獻大多數借閱（冪次權重）；
  2. **學期節律**：開學與期末月借閱多、寒暑假少（月權重）＋平日多於週末（週權重）；
  3. **右偏**：多數人準時還，少數人拖很久（借閱天數用 lognormal，逾期是右尾）；
  4. **相關**：熱門書更容易「冊冊借光」→ 預約集中在熱門書（用同一組熱門度權重抽）。
- **一致性**：`available_copies ＝ total_copies − 在外未還筆數`，生成完立刻對帳（設計決策 1 的驗算）。

In [ ]:
#@title 🎲 合成 books（800）與 members（500）——固定 seed=42
rng = np.random.default_rng(42)

CATS = np.array(['統計','數學','資訊','小說','商管','科普','語言','漫畫'])
CAT_W = np.array([.18,.12,.16,.18,.08,.12,.06,.10])
TITLE_POOL = {'統計':['迴歸分析實戰','貝氏思維','抽樣的藝術','統計學習導論','時間序列圖解','實驗設計入門'],
              '數學':['線性代數之美','微積分的日常','機率論漫步','離散數學圖鑑'],
              '資訊':['資料庫概論','演算法圖解','Python 資料科學','系統設計入門','SQL 進階之路'],
              '小說':['深夜圖書館','借閱人生','紙上偵探','海風書店','雨季不再來'],
              '商管':['管理的常識','行銷資料學','財報狗都懂'],
              '科普':['宇宙的尺度','基因藍圖','氣候方程式','大腦簡史'],
              '語言':['日語五十音樂','英文寫作課'],
              '漫畫':['統計少女','資料庫勇者','圖書館戰記']}
SURNAMES = list("陳林黃張李王吳劉蔡楊許鄭謝郭洪曾邱廖賴周")
GIVEN = ["佳蓉","威廷","雅筑","承翰","思穎","冠宇","孟軒","子涵","明修","芷瑄",
         "宇翔","欣妤","偉倫","品妍","柏勳","韻如","家豪","美慧","國彬","語彤"]

N_BOOK, N_MEM = 800, 500
book_cats = rng.choice(CATS, N_BOOK, p=CAT_W)
books = pd.DataFrame({
    "book_id": np.arange(1, N_BOOK + 1),
    "title": [f"{rng.choice(TITLE_POOL[c])}（{i:03d}）" for i, c in enumerate(book_cats, 1)],  # 尾碼保唯一
    "author": [rng.choice(SURNAMES) + rng.choice(GIVEN) for _ in range(N_BOOK)],
    "isbn": [f"978-957-{i:05d}-{rng.integers(10):d}" for i in range(1, N_BOOK + 1)],
    "category": book_cats,
    "total_copies": rng.choice([1, 2, 3, 4, 5], N_BOOK, p=[.30, .30, .20, .12, .08]),
})
members = pd.DataFrame({
    "member_id": np.arange(1, N_MEM + 1),
    "name": [rng.choice(SURNAMES) + rng.choice(GIVEN) for _ in range(N_MEM)],
    "email": [f"m{i:04d}@stat.example.edu" for i in range(1, N_MEM + 1)],
    "joined": [str(np.datetime64('2025-09-01') + np.timedelta64(int(d), 'D'))
               for d in rng.integers(0, 300, N_MEM)],
})
# 長尾熱門度：第 k 熱門的書，被借機率 ∝ 1/k^0.8（洗牌讓熱門書散在各類）
rank = rng.permutation(N_BOOK) + 1
pop_w = 1 / rank ** 0.8
pop_w = pop_w / pop_w.sum()
print(f"books {len(books)} 列、members {len(members)} 列；最熱門 5% 的書將吃下 "
      f"{pop_w[np.argsort(-pop_w)][:N_BOOK//20].sum()*100:.0f}% 的借閱量（長尾）")

In [ ]:
#@title 🎲 合成 loans（≈13,000：歷史已還＋在外未還）與 reservations——並維持庫存一致
# --- (a) 歷史已還 12,000 筆 ---
N_HIST = 12_000
today = np.datetime64(DEMO_TODAY)
start_d = np.datetime64('2025-09-01')
n_days = int((today - start_d).astype(int)) - 28          # 歷史一路生成到模擬今日前 28 天
day_idx = np.arange(n_days)
month_idx = ((day_idx // 30) % 12)
month_w = np.array([1.5,1.2,1.0,1.6,1.7,0.5,0.9,1.3,1.2,1.6,1.4,0.6])[month_idx % 12]  # 開學/期末高、假期低
weekdays = pd.to_datetime((start_d + day_idx.astype("timedelta64[D]")).astype(str)).dayofweek.to_numpy()
week_w = np.array([1.2,1.2,1.1,1.1,1.0,0.6,0.5])[weekdays]                            # 真實星期：平日 > 週末
day_w = month_w * week_w; day_w = day_w / day_w.sum()

loan_days = start_d + rng.choice(n_days, N_HIST, p=day_w).astype("timedelta64[D]")
loan_books = rng.choice(books.book_id.to_numpy(), N_HIST, p=pop_w)
loan_members = rng.integers(1, N_MEM + 1, N_HIST)
borrow_days = np.clip(np.round(rng.lognormal(np.log(16), 0.55, N_HIST)), 1, 90).astype(int)  # 右偏：中位約 16 天
return_days = np.minimum(loan_days + borrow_days.astype("timedelta64[D]"), today)
hist = pd.DataFrame({
    "book_id": loan_books, "member_id": loan_members,
    "loan_date": loan_days.astype(str),
    "due_date": (loan_days + np.timedelta64(28, "D")).astype(str),
    "return_date": return_days.astype(str),
    "renew_count": rng.choice([0, 1, 2], N_HIST, p=[.72, .2, .08]),
})
# --- (b) 在外未還：逐書生成，每本 ≤ 總冊數 → available 天生一致 ---
out_rows = []
for bid, copies, w in zip(books.book_id, books.total_copies, pop_w * N_BOOK):
    n_out = min(copies, rng.poisson(0.32 * min(w, 3.0)))          # 熱門書更容易被借光
    for m in rng.choice(N_MEM, n_out, replace=False):
        d = today - np.timedelta64(int(rng.integers(1, 55)), "D") # 最近 55 天內借的 → 部分已逾期
        out_rows.append((bid, int(m) + 1, str(d), str(d + np.timedelta64(28, "D")), None, 0))
outs = pd.DataFrame(out_rows, columns=hist.columns)
loans = pd.concat([hist, outs], ignore_index=True)
loans.insert(0, "loan_id", np.arange(1, len(loans) + 1))

# --- (c) 寫入 SQLite（executemany ＋ 單一交易：U05 的效能鐵則） ---
with lcon:
    lcon.executemany("INSERT INTO member VALUES (?,?,?,?)", members.astype(object).itertuples(index=False))
    n_out_map = outs.groupby("book_id").size()
    books["available_copies"] = (books.total_copies - books.book_id.map(n_out_map).fillna(0)).astype(int)
    lcon.executemany("INSERT INTO book VALUES (?,?,?,?,?,?,?)", books.astype(object).itertuples(index=False))
    lcon.executemany("INSERT INTO loan VALUES (?,?,?,?,?,?,?)",
                     loans.astype(object).where(pd.notna(loans), None).itertuples(index=False))
# --- (d) 預約：集中在「目前借不到」的熱門書 ---
no_stock = books[books.available_copies == 0].book_id.to_numpy()
resv = []
rid = 1
for bid in no_stock:
    for m in rng.choice(N_MEM, rng.integers(0, 4), replace=False):
        st = rng.choice(['等待', '取消', '完成'], p=[.6, .25, .15])
        resv.append((rid, int(bid), int(m) + 1,
                     str(today - np.timedelta64(int(rng.integers(0, 20)), "D")) + " 12:00", st)); rid += 1
with lcon:
    lcon.executemany("INSERT INTO reservation VALUES (?,?,?,?,?)", resv)

for t in ["book", "member", "loan", "reservation"]:
    print(f"{t:12s}{lcon.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>7,} 列")
total_rows = sum(lcon.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                 for t in ["book", "member", "loan", "reservation"])
assert total_rows >= 10_000, "共同要求 2：至少一萬列"
print(f"合計 {total_rows:,} 列 ✅（共同要求 2 達標）")

In [ ]:
# 模擬時鐘回歸檢查：歷史資料不可穿越 DEMO_TODAY
n_future = lcon.execute("""SELECT COUNT(*) FROM loan
                            WHERE loan_date > ?
                               OR (return_date IS NOT NULL AND return_date > ?)""",
                        (DEMO_TODAY, DEMO_TODAY)).fetchone()[0]
print("晚於模擬今日的歷史日期：", n_future)
assert n_future == 0
print("✅ 所有借閱歷史都停在 DEMO_TODAY 以前")

In [ ]:
# 對帳（設計決策 1 的驗算）：快照欄 available_copies ≟ 總冊數 − 在外未還
n_bad = lcon.execute("""
    SELECT COUNT(*) FROM book b
    WHERE b.available_copies <>
          b.total_copies - (SELECT COUNT(*) FROM loan l
                            WHERE l.book_id = b.book_id AND l.return_date IS NULL)""").fetchone()[0]
print("快照 vs 即算 不一致筆數：", n_bad)
assert n_bad == 0
print("✅ 庫存一致（之後每次借/還都由交易維護；這句對帳查詢保留著，隨時可稽核）")

In [ ]:
#@title 🈶 圖表中文字型（Colab 需安裝一次；本機有 Noto 就直接生效）
import os, glob, subprocess, matplotlib
from matplotlib import font_manager

cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if not cjk:                                    # Colab 第一次：裝字型（約 10 秒）
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], capture_output=True)
    cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if cjk:
    font_manager.fontManager.addfont(cjk[0])
    matplotlib.rcParams["font.family"] = font_manager.FontProperties(fname=cjk[0]).get_name()
    print("中文字型就緒 ✅：", os.path.basename(cjk[0]))
else:
    print("⚠️ 找不到 CJK 字型——圖表中文會變 □（不影響其他功能）")
matplotlib.rcParams["axes.unicode_minus"] = False

In [ ]:
# 分佈自我檢查：長尾與學期節律「長得像不像真的」（報告裡放這兩張圖，評分官最愛）
import matplotlib.pyplot as plt

df_loans = pd.read_sql_query("SELECT book_id, loan_date FROM loan", lcon)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
df_loans.book_id.value_counts().reset_index(drop=True).plot(ax=axes[0], lw=1.5)
axes[0].set_title("每本書借閱次數（由熱到冷排序）"); axes[0].set_xlabel("書的熱門名次"); axes[0].set_ylabel("借閱次數")
pd.to_datetime(df_loans.loan_date).dt.to_period("M").astype(str).value_counts().sort_index() \
    .plot.bar(ax=axes[1], width=0.8)
axes[1].set_title("每月借閱量"); axes[1].tick_params(axis='x', rotation=60, labelsize=7)
plt.tight_layout(); plt.show()
print("左：少數書吃掉大多數借閱（長尾 ✅）；右：期末月尖峰、寒暑假谷底（節律 ✅）")

# §4 資料層：CRUD 函數（共同要求 3、4）

原則（U05）：**UI 薄、邏輯厚**——每個函數都：值走 `?`、驗證輸入、回傳人話訊息、**寫操作包交易**、
寫完馬上配一個 assert。函數清單：

| 類 | 函數 | 交易？ |
|---|---|---|
| C | `add_book` `register_member` `reserve_book` | ✔ |
| R | `search_books` `member_history` `overdue_list` | — |
| U | `borrow_book` `return_book` `renew_loan` `adjust_copies` | ✔（借還是**核心業務操作**） |
| D | `delete_book`（有外借要擋） | ✔ |

In [ ]:
# R：查書（關鍵字＋類別；空字串＝全部）——櫃台與後台共用
def search_books(keyword="", category="全部", only_avail=False, limit=30):
    sql = """SELECT book_id AS 編號, title AS 書名, author AS 作者, category AS 類別,
                    available_copies AS 可借, total_copies AS 總冊數
             FROM book
             WHERE (title LIKE ? OR author LIKE ?)"""
    params = [f"%{keyword}%", f"%{keyword}%"]
    if category != "全部":
        sql += " AND category = ?"; params.append(category)
    if only_avail:
        sql += " AND available_copies > 0"
    sql += " ORDER BY book_id LIMIT ?"; params.append(limit)
    return pd.read_sql_query(sql, lcon, params=params)

print(search_books("統計", "全部").head(3).to_string(index=False))
assert len(search_books("絕對沒有這本書XYZ")) == 0 and len(search_books("")) > 0
print("✅ search_books OK（關鍵字、類別、只看可借三個條件都吃 ? 參數）")

In [ ]:
# C：辦證（email 唯一由約束把關——程式只負責把錯誤翻成人話）
def register_member(name, email, joined=DEMO_TODAY):
    name, email = (name or "").strip(), (email or "").strip().lower()
    if not name or "@" not in email:
        return "⚠️ 姓名必填、email 要像 email"
    try:
        joined = validate_iso_date(joined, "辦證日期")
        with lcon:
            cur = lcon.execute("INSERT INTO member(name, email, joined) VALUES (?,?,?)",
                               (name, email, joined))
        return f"✅ 辦證成功，會員編號 {cur.lastrowid}"
    except ValueError as e:
        return f"⚠️ {e}"
    except sqlite3.IntegrityError:
        return "❌ 這個 email 已經辦過證了"

print(register_member("測試員", "demo@stat.example.edu"))
print(register_member("重複的", "demo@stat.example.edu"))          # 應該失敗 → 友善訊息
assert register_member("", "x").startswith("⚠️")
print("✅ register_member OK（成功、重複、缺欄三情境）")

In [ ]:
# C＋U：新書入館／調整冊數（調整要同時動 total 與 available，包在交易裡）
def add_book(title, author, category, copies, isbn=None):
    title = (title or "").strip()
    if not title or not (author or "").strip():
        return "⚠️ 書名與作者必填"
    try:
        copies = int(copies)
        with lcon:
            cur = lcon.execute("""INSERT INTO book(title, author, isbn, category,
                                   total_copies, available_copies) VALUES (?,?,?,?,?,?)""",
                               (title, author, isbn, category, copies, copies))
        return f"✅ 入館成功，書目編號 {cur.lastrowid}（{copies} 冊）"
    except (ValueError, sqlite3.IntegrityError) as e:
        return f"❌ 入館失敗：{e}"

def adjust_copies(book_id, delta):
    try:
        with lcon:
            n = lcon.execute("""UPDATE book SET total_copies = total_copies + ?,
                                available_copies = available_copies + ? WHERE book_id = ?""",
                             (delta, delta, book_id)).rowcount
        return "✅ 已調整" if n else "⚠️ 查無此書"
    except sqlite3.IntegrityError:
        return "❌ 調整後會違反 0 ≤ 可借 ≤ 總冊數（有書在外借時不能砍到比在外冊數少）"

print(add_book("SQL 佔位符之歌", "王教授", "資訊", 2))
print(adjust_copies(1, +1))
print(adjust_copies(1, -99))                              # 應該失敗：CHECK 擋（在外借的收不回來）
assert add_book("", "", "統計", 1).startswith("⚠️")
assert adjust_copies(1, -99).startswith("❌")
print("✅ add_book／adjust_copies OK（約束當最後防線，訊息翻成人話）")

In [ ]:
# U（核心業務操作①）：借書——「檢查可借 → 扣庫存 → 開借閱單」必須原子完成
def borrow_book(member_id, book_id, on_date=DEMO_TODAY):
    try:
        on_date = validate_iso_date(on_date, "借書日期")
        due_date = (date.fromisoformat(on_date) + timedelta(days=28)).isoformat()
        with lcon:                                        # 同生共死：任何一步失敗，全部回滾
            mem = lcon.execute("SELECT name FROM member WHERE member_id = ?", (member_id,)).fetchone()
            if mem is None:
                raise ValueError("查無此會員")
            # 條件式 UPDATE：把「檢查」和「扣」合成一步（§5 會演示為什麼這樣寫）
            n = lcon.execute("""UPDATE book SET available_copies = available_copies - 1
                                WHERE book_id = ? AND available_copies >= 1""", (book_id,)).rowcount
            if n == 0:
                raise ValueError("這本書目前 0 冊可借（可用「預約」排隊）")
            cur = lcon.execute("""INSERT INTO loan(book_id, member_id, loan_date, due_date)
                                  VALUES (?, ?, ?, ?)""", (book_id, member_id, on_date, due_date))
        return f"✅ 借出成功，借閱單 #{cur.lastrowid}，到期日 {due_date}"
    except ValueError as e:
        return f"❌ {e}"

bid_avail = lcon.execute("SELECT book_id FROM book WHERE available_copies > 0 ORDER BY book_id LIMIT 1").fetchone()[0]
print(f"對 {bid_avail} 號書借書：", borrow_book(1, bid_avail))
r = lcon.execute("SELECT available_copies, total_copies FROM book WHERE book_id = ?", (bid_avail,)).fetchone()
print(f"{bid_avail} 號書可借 {r[0]}/{r[1]}")
assert borrow_book(99999, 1).startswith("❌")             # 應該失敗：幽靈會員
print("✅ borrow_book OK（成功＋幽靈會員被擋）")

In [ ]:
# U（核心業務操作②）：還書——「結案 ＋ 回庫 ＋ 喚醒最早的預約」三件事一個交易
def return_book(loan_id, on_date=DEMO_TODAY):
    try:
        on_date = validate_iso_date(on_date, "還書日期")
        with lcon:
            l = lcon.execute("""SELECT book_id FROM loan
                                WHERE loan_id = ? AND return_date IS NULL""", (loan_id,)).fetchone()
            if l is None:
                raise ValueError("查無此未還借閱單")
            lcon.execute("UPDATE loan SET return_date = ? WHERE loan_id = ?", (on_date, loan_id))
            lcon.execute("UPDATE book SET available_copies = available_copies + 1 WHERE book_id = ?",
                         (l["book_id"],))
            next_res = lcon.execute("""SELECT res_id, member_id FROM reservation
                                       WHERE book_id = ? AND status = '等待'
                                       ORDER BY res_time LIMIT 1""", (l["book_id"],)).fetchone()
            notice = ""
            if next_res:
                lcon.execute("UPDATE reservation SET status = '已通知' WHERE res_id = ?",
                             (next_res["res_id"],))
                notice = f"；📣 已通知預約會員 #{next_res['member_id']} 來取書"
        return f"✅ 還書完成{notice}"
    except ValueError as e:
        return f"❌ {e}"
    except sqlite3.IntegrityError:
        return "❌ 還書日期不能早於借書日期"

open_loan = lcon.execute("SELECT loan_id FROM loan WHERE return_date IS NULL LIMIT 1").fetchone()[0]
print(f"拿一張未還單 #{open_loan} 來還：", return_book(open_loan))
assert return_book(open_loan).startswith("❌")             # 應該失敗：同一張單還兩次
print("✅ return_book OK（結案＋回庫＋喚醒預約是同一個交易；還兩次被擋）")

In [ ]:
# U：續借（規則：≤2 次、無人等待）；C：預約（部分唯一索引擋重複）；D：刪書（有外借擋）
def renew_loan(loan_id):
    try:
        with lcon:
            l = lcon.execute("""SELECT book_id, renew_count FROM loan
                                WHERE loan_id = ? AND return_date IS NULL""", (loan_id,)).fetchone()
            if l is None:
                raise ValueError("查無此未還借閱單")
            if l["renew_count"] >= 2:
                raise ValueError("已續借 2 次，不能再續")
            n_waiting = lcon.execute("""SELECT COUNT(*) FROM reservation
                                        WHERE book_id = ? AND status = '等待'""",
                                     (l["book_id"],)).fetchone()[0]
            if n_waiting:
                raise ValueError("有人正在預約這本書，不能續借")
            lcon.execute("""UPDATE loan SET due_date = date(due_date, '+28 day'),
                            renew_count = renew_count + 1 WHERE loan_id = ?""", (loan_id,))
        return "✅ 續借成功，到期日 +28 天"
    except ValueError as e:
        return f"❌ {e}"

def reserve_book(member_id, book_id, on_date=DEMO_TODAY):
    try:
        on_date = validate_iso_date(on_date, "預約日期")
        with lcon:
            avail_row = lcon.execute("SELECT available_copies FROM book WHERE book_id = ?",
                                     (book_id,)).fetchone()
            if avail_row is None:
                raise ValueError("查無此書")
            if avail_row[0] > 0:
                raise ValueError("這本書現在就借得到，直接借！")
            lcon.execute("""INSERT INTO reservation(book_id, member_id, res_time)
                              VALUES (?,?,?)""", (book_id, member_id, f"{on_date} 12:00:00"))
        return "✅ 已排入預約"
    except ValueError as e:
        return f"❌ {e}"
    except sqlite3.IntegrityError:
        return "❌ 你已經在等這本書了（等待中不可重複預約）"

def delete_book(book_id):
    try:
        with lcon:
            n_out = lcon.execute("""SELECT COUNT(*) FROM loan
                                    WHERE book_id = ? AND return_date IS NULL""", (book_id,)).fetchone()[0]
            if n_out:
                raise ValueError(f"還有 {n_out} 冊在外未還，不能下架")
            lcon.execute("DELETE FROM reservation WHERE book_id = ?", (book_id,))
            n = lcon.execute("DELETE FROM book WHERE book_id = ?", (book_id,)).rowcount
        return "✅ 已下架" if n else "⚠️ 查無此書"
    except ValueError as e:
        return f"❌ {e}"
    except sqlite3.IntegrityError:
        return "❌ 這本書有歷史借閱紀錄，請保留書目並改用下架狀態（軟刪）"

bid_empty = lcon.execute("""SELECT b.book_id FROM book b
    WHERE b.available_copies = 0
      AND NOT EXISTS (SELECT 1 FROM reservation r
                      WHERE r.book_id = b.book_id AND r.member_id = 2 AND r.status = '等待')
    ORDER BY b.book_id LIMIT 1""").fetchone()[0]
print("預約沒貨書：", reserve_book(2, bid_empty))
print("重複預約　：", reserve_book(2, bid_empty))          # 部分唯一索引出手
print("續借剛剛那張已還單：", renew_loan(open_loan))
bid_out = lcon.execute("SELECT book_id FROM loan WHERE return_date IS NULL LIMIT 1").fetchone()[0]
print(f"刪 {bid_out} 號書（有外借）：", delete_book(bid_out))
bid_history = lcon.execute("""SELECT b.book_id FROM book b
    WHERE EXISTS (SELECT 1 FROM loan l WHERE l.book_id = b.book_id)
      AND NOT EXISTS (SELECT 1 FROM loan l WHERE l.book_id = b.book_id AND l.return_date IS NULL)
    ORDER BY b.book_id LIMIT 1""").fetchone()[0]
history_msg = delete_book(bid_history)
print(f"刪 {bid_history} 號書（只有歷史借閱）：", history_msg)
assert "歷史借閱紀錄" in history_msg
print("✅ renew／reserve／delete OK（每條業務規則都有對應的擋法）")

In [ ]:
# R：兩個櫃台常用查詢——會員借閱史（join）、逾期清單（date 運算＋join）
def member_history(member_id):
    return pd.read_sql_query("""
        SELECT l.loan_id AS 單號, b.title AS 書名, l.loan_date AS 借出, l.due_date AS 到期,
               COALESCE(l.return_date, '（在外）') AS 歸還, l.renew_count AS 續借
        FROM loan l JOIN book b ON l.book_id = b.book_id
        WHERE l.member_id = ?
        ORDER BY l.loan_date DESC LIMIT 20""", lcon, params=(member_id,))

def overdue_list(as_of=DEMO_TODAY):
    as_of = validate_iso_date(as_of, "查詢基準日")
    return pd.read_sql_query("""
        SELECT m.name AS 讀者, m.email, b.title AS 書名, l.due_date AS 到期日,
               CAST(julianday(?) - julianday(l.due_date) AS INTEGER) AS 逾期天數
        FROM loan l
        JOIN member m ON l.member_id = m.member_id
        JOIN book b   ON l.book_id  = b.book_id
        WHERE l.return_date IS NULL AND l.due_date < ?
        ORDER BY 逾期天數 DESC""", lcon, params=(as_of, as_of))

print(member_history(1).head(3).to_string(index=False))
print(f"\n目前逾期 {len(overdue_list())} 筆；最兇的三筆：")
print(overdue_list().head(3).to_string(index=False))

# §5 併發競態：兩個人同時搶最後一本書（共同要求 4 的重頭戲）

> 【導讀】§4 的 `borrow_book()` 裡那句 `UPDATE ... WHERE available_copies >= 1` 為什麼不寫成
> 「先 SELECT 檢查、再 UPDATE 扣一」？因為**兩個人可以同時通過檢查**。
> 下面開兩條連線扮演兩個同時按下「借書」的人——先演壞的寫法。

壞寫法（check-then-act）：`SELECT` 看到有貨 → `UPDATE` 扣一。兩人**都**看到「還有 1 本」，就都扣——超賣。

In [ ]:
# 先重現事故：用一張「沒有 CHECK 保護」的影子櫃檯，跑 check-then-act
lcon.executescript("""
DROP TABLE IF EXISTS shadow_book;
CREATE TABLE shadow_book(book_id INTEGER PRIMARY KEY, title TEXT, available INTEGER);  -- 故意不設 CHECK
INSERT INTO shadow_book VALUES (1, '絕版的統計神書', 1);                                -- 只剩最後一本！
"""); lcon.commit()

con_a = sqlite3.connect("library.db"); con_b = sqlite3.connect("library.db")   # 兩個使用者＝兩條連線

a_sees = con_a.execute("SELECT available FROM shadow_book WHERE book_id = 1").fetchone()[0]
b_sees = con_b.execute("SELECT available FROM shadow_book WHERE book_id = 1").fetchone()[0]
print(f"甲查詢：剩 {a_sees} 本 → 決定借！")
print(f"乙查詢：剩 {b_sees} 本 → 也決定借！（兩人都通過了『檢查』）")

con_a.execute("UPDATE shadow_book SET available = available - 1 WHERE book_id = 1"); con_a.commit()
con_b.execute("UPDATE shadow_book SET available = available - 1 WHERE book_id = 1"); con_b.commit()
left = lcon.execute("SELECT available FROM shadow_book WHERE book_id = 1").fetchone()[0]
print(f"\n兩人都『成功』借走 → 庫存變成 {left}！！同一本書借給兩個人（超賣）")
assert left == -1
con_a.close(); con_b.close()
print("→ 注意：兩個人的程式碼都沒寫錯，是「檢查」與「動作」之間有縫隙。")

In [ ]:
# 防護版：條件式 UPDATE——「檢查」和「扣」合成同一句，資料庫保證一次只有一人改
lcon.execute("UPDATE shadow_book SET available = 1 WHERE book_id = 1"); lcon.commit()   # 重新上架最後一本

con_a = sqlite3.connect("library.db"); con_b = sqlite3.connect("library.db")
a_got = con_a.execute("""UPDATE shadow_book SET available = available - 1
                          WHERE book_id = 1 AND available >= 1""").rowcount; con_a.commit()
b_got = con_b.execute("""UPDATE shadow_book SET available = available - 1
                          WHERE book_id = 1 AND available >= 1""").rowcount; con_b.commit()
left = lcon.execute("SELECT available FROM shadow_book WHERE book_id = 1").fetchone()[0]
print(f"甲的 UPDATE 改到 {a_got} 列 → {'✅ 借到了' if a_got else '❌ 沒搶到'}")
print(f"乙的 UPDATE 改到 {b_got} 列 → {'✅ 借到了' if b_got else '❌ 沒搶到 → 引導去預約'}")
print(f"庫存 = {left}（不會是負的）")
assert (a_got, b_got, left) == (1, 0, 0)
con_a.close(); con_b.close()
print()
print("→ rowcount 告訴你「搶到沒」；沒搶到就走預約流程——這正是 §4 borrow_book() 的寫法。")
print("  正式表還有 CHECK(available BETWEEN 0 AND total) 當最後防線：就算程式寫錯也擋在 -1 之前。")
print("  （更多武器：BEGIN IMMEDIATE 直接鎖住整段交易——U06 課堂拆解。）")

# §6 統計報表 ×6（共同要求 5：≥1 window、≥2 join、≥1 圖表）

每張報表先一句「**它回答什麼問題**」——這句話之後就是你簡報裡的標題。

In [ ]:
# 報表 1【哪些書最熱門？】join ＋ 聚合（採購與汰舊的依據）
pd.read_sql_query("""
    SELECT b.title AS 書名, b.category AS 類別, COUNT(*) AS 借閱次數,
           b.total_copies AS 冊數,
           ROUND(COUNT(*) * 1.0 / b.total_copies, 1) AS 每冊周轉
    FROM loan l JOIN book b ON l.book_id = b.book_id
    GROUP BY b.book_id ORDER BY 借閱次數 DESC LIMIT 10""", lcon)

In [ ]:
# 報表 2【借閱量的趨勢？】月借閱量＋3 月移動平均（window frame）——給館方看淡旺季
pd.read_sql_query("""
    WITH m AS (SELECT strftime('%Y-%m', loan_date) AS ym, COUNT(*) AS n
               FROM loan GROUP BY ym)
    SELECT ym AS 月份, n AS 借閱量,
           ROUND(AVG(n) OVER (ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1)
               AS 近三月移動平均
    FROM m ORDER BY ym""", lcon).tail(8)

In [ ]:
# 報表 3【誰是重度讀者？】NTILE(4) 讀者活躍度分級（window）——經營分眾的起點
pd.read_sql_query("""
    WITH t AS (SELECT m.member_id, m.name, COUNT(l.loan_id) AS n
               FROM member m LEFT JOIN loan l ON m.member_id = l.member_id
               GROUP BY m.member_id),
    g AS (SELECT *, NTILE(4) OVER (ORDER BY n) AS tier FROM t)
    SELECT tier AS 活躍度四分位, COUNT(*) AS 人數,
           MIN(n) AS 最少借閱, MAX(n) AS 最多借閱, ROUND(AVG(n),1) AS 平均
    FROM g GROUP BY tier ORDER BY tier""", lcon)

In [ ]:
# 報表 4【逾期壓力多大？】逾期中明細（雙 join ＋ 日期運算）——催還名單直接匯出
print(f"目前逾期 {len(overdue_list())} 筆")
overdue_list().head(8)

In [ ]:
# 報表 5【各類書的淡旺季？】類別 × 月份 樞紐（條件式聚合）——排架與採購的節奏
pd.read_sql_query("""
    SELECT strftime('%Y-%m', loan_date) AS 月份,
           SUM(category = '統計') AS 統計, SUM(category = '資訊') AS 資訊,
           SUM(category = '小說') AS 小說, SUM(category = '漫畫') AS 漫畫,
           COUNT(*) AS 全部
    FROM loan l JOIN book b ON l.book_id = b.book_id
    GROUP BY 月份 ORDER BY 月份 DESC LIMIT 8""", lcon)

In [ ]:
# 報表 6【圖表版】月借閱趨勢＋熱門類別佔比——這兩個函數等下直接嵌進 gr.Plot
def trend_chart():
    df = pd.read_sql_query("""SELECT strftime('%Y-%m', loan_date) AS ym, COUNT(*) AS n
                              FROM loan GROUP BY ym ORDER BY ym""", lcon)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(df.ym, df.n, marker="o", ms=3)
    ax.set_title("每月借閱量趨勢"); ax.tick_params(axis='x', rotation=60, labelsize=7)
    plt.tight_layout(); return fig

def category_chart():
    df = pd.read_sql_query("""SELECT b.category AS c, COUNT(*) AS n
                              FROM loan l JOIN book b ON l.book_id = b.book_id
                              GROUP BY c ORDER BY n DESC""", lcon)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(df.c, df.n)
    ax.set_title("各類別借閱量")
    plt.tight_layout(); return fig

for f in (trend_chart, category_chart):
    fig = f(); print(f.__name__, "→", type(fig).__name__); plt.close(fig)
print("✅ 報表 1–6 齊：window ×2（報表 2、3）、join ×4、圖表 ×2 —— 共同要求 5 達標")

# §7 效能：最慢的查詢 → EXPLAIN → 建索引 → 前後計時（共同要求 6）

> 【導讀】系統最常按的按鈕是「查某會員的借閱史」。loan 有一萬三千列，每按一次都整表掃。
> 讓查詢計畫自己招供，然後用一個索引讓它閉嘴——這就是共同要求 6 要的四行證據。

In [ ]:
import time
q_sql = "SELECT * FROM loan WHERE member_id = ?"

print("── 建索引前的查詢計畫 ──")
for r in lcon.execute(f"EXPLAIN QUERY PLAN {q_sql}", (123,)):
    print("  ", r[3])                                     # SCAN loan ＝ 整表掃描

t = time.perf_counter()
for i in range(2000):
    lcon.execute(q_sql, (i % 500 + 1,)).fetchall()
t_slow = time.perf_counter() - t

lcon.execute("CREATE INDEX IF NOT EXISTS idx_loan_member ON loan(member_id)")

print("\n── 建 idx_loan_member 之後 ──")
for r in lcon.execute(f"EXPLAIN QUERY PLAN {q_sql}", (123,)):
    print("  ", r[3])                                     # SEARCH ... USING INDEX

t = time.perf_counter()
for i in range(2000):
    lcon.execute(q_sql, (i % 500 + 1,)).fetchall()
t_fast = time.perf_counter() - t

print(f"\n2,000 次查詢：{t_slow*1000:.0f} ms → {t_fast*1000:.0f} ms（快 {t_slow/t_fast:.0f} 倍）")
assert t_fast < t_slow
print("✅ 共同要求 6 達標。資料量越大差距越誇張——原理 U07 拆解（B+ tree）")

# §8 Gradio 三分頁介面（共同要求 7）

分頁規劃（照 projects.md 建議的「日常操作／後台管理／統計報表」）：

| 分頁 | 給誰用 | 內容 |
|---|---|---|
| 🏷 流通櫃台 | 館員日常 | 借書（會員＋書名關鍵字）、還書（選單只列未還單）、逾期清單 |
| ⚙️ 後台管理 | 館長 | 新書入館、辦證、查書 |
| 📊 統計報表 | 決策 | 報表下拉選單 ＋ 兩張圖 |

介面頂端的「業務日期」是**三個分頁共用**的模擬時鐘：借還、辦證與逾期報表都讀同一個值。

注意兩個 U05 的招式：**還書下拉選單跟著會員變**（`gr.update`）、**所有表格都即時重撈**。

In [ ]:
import sys, importlib.util, subprocess
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr

member_choices = lambda: [(f"{r['name']}（#{r['member_id']}）", r["member_id"])
                          for r in lcon.execute("SELECT member_id, name FROM member ORDER BY member_id")]
avail_choices = lambda: [(f"{r['title']}（剩 {r['available_copies']}）", r["book_id"])
                         for r in lcon.execute("""SELECT book_id, title, available_copies FROM book
                                                  WHERE available_copies > 0 ORDER BY book_id""")]
def open_loan_choices(member_id):
    rows = lcon.execute("""SELECT l.loan_id, b.title FROM loan l JOIN book b ON l.book_id = b.book_id
                           WHERE l.member_id = ? AND l.return_date IS NULL""", (member_id,)).fetchall()
    return gr.update(choices=[(f"#{r['loan_id']} {r['title']}", r["loan_id"]) for r in rows], value=None)

def ui_borrow(member_id, book_id, on_date):
    if member_id is None or book_id is None:
        return "⚠️ 會員與書都要選", gr.update(), gr.update()
    return borrow_book(member_id, book_id, on_date), gr.update(choices=avail_choices()), open_loan_choices(member_id)

def ui_return(member_id, loan_id, on_date):
    if loan_id is None:
        return "⚠️ 先選要還的單", gr.update(), gr.update()
    return return_book(loan_id, on_date), gr.update(choices=avail_choices()), open_loan_choices(member_id)

print("UI 包裝函數就緒（薄薄一層：驗證＋呼叫資料層＋刷新選單）")

In [ ]:
REPORTS = {
    "熱門書 Top 10":  """SELECT b.title 書名, COUNT(*) 借閱次數 FROM loan l JOIN book b ON l.book_id=b.book_id
                        GROUP BY b.book_id ORDER BY 2 DESC LIMIT 10""",
    "月借閱量＋移動平均": """WITH m AS (SELECT strftime('%Y-%m', loan_date) ym, COUNT(*) n FROM loan GROUP BY ym)
                        SELECT ym 月份, n 借閱量, ROUND(AVG(n) OVER (ORDER BY ym
                        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),1) 近三月平均 FROM m ORDER BY ym""",
    "讀者活躍度分級":  """WITH t AS (SELECT m.member_id, COUNT(l.loan_id) n FROM member m
                        LEFT JOIN loan l ON m.member_id=l.member_id GROUP BY m.member_id),
                        g AS (SELECT *, NTILE(4) OVER (ORDER BY n) tier FROM t)
                        SELECT tier 分級, COUNT(*) 人數, MIN(n) 最少, MAX(n) 最多 FROM g GROUP BY tier""",
    "逾期中清單":     """SELECT m.name 讀者, b.title 書名, l.due_date 到期,
                        CAST(julianday(?) - julianday(l.due_date) AS INTEGER) 逾期天
                        FROM loan l JOIN member m ON l.member_id=m.member_id
                        JOIN book b ON l.book_id=b.book_id
                        WHERE l.return_date IS NULL AND l.due_date < ?
                        ORDER BY 4 DESC LIMIT 30""",
}
def run_report(name, as_of=DEMO_TODAY):
    as_of = validate_iso_date(as_of, "報表基準日")
    params = (as_of, as_of) if name == "逾期中清單" else ()
    return pd.read_sql_query(REPORTS[name], lcon, params=params)

assert len(run_report("熱門書 Top 10")) == 10             # 報表函數也先測
print("報表選單就緒：", list(REPORTS))

In [ ]:
# 三分頁組裝（這格就是你專題 notebook 倒數第二格的樣子）
with gr.Blocks(title="圖書館借閱管理") as app:
    gr.Markdown("# 📚 圖書館借閱管理系統（教師示範專題）")
    gr.Markdown("下方「業務日期」是三個分頁共用的模擬時鐘；一律輸入 `YYYY-MM-DD`。")
    txt_business_date = gr.Textbox(value=DEMO_TODAY, label="業務日期（三分頁共用；YYYY-MM-DD）")
    with gr.Tab("🏷 流通櫃台"):
        with gr.Row():
            dd_member = gr.Dropdown(choices=member_choices(), label="會員", value=1, filterable=True)
            dd_book   = gr.Dropdown(choices=avail_choices(), label="借書（只列可借）", filterable=True)
            dd_open   = gr.Dropdown(choices=[], label="還書（只列此會員未還單）")
        msg = gr.Textbox(label="結果", interactive=False)
        with gr.Row():
            gr.Button("借出", variant="primary").click(
                ui_borrow, [dd_member, dd_book, txt_business_date], [msg, dd_book, dd_open])
            gr.Button("歸還").click(
                ui_return, [dd_member, dd_open, txt_business_date], [msg, dd_book, dd_open])
        dd_member.change(open_loan_choices, [dd_member], [dd_open])   # 換會員 → 未還單選單跟著換（gr.update）
        gr.Markdown("### ⏰ 逾期清單（每次打開自動更新）")
        tbl_overdue = gr.Dataframe(value=overdue_list(DEMO_TODAY))
        gr.Button("依業務日期重新整理").click(overdue_list, txt_business_date, tbl_overdue)
    with gr.Tab("⚙️ 後台管理"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("#### 新書入館")
                t1 = gr.Textbox(label="書名"); t2 = gr.Textbox(label="作者")
                t3 = gr.Dropdown(['統計','數學','資訊','小說','商管','科普','語言','漫畫'],
                                 value='統計', label="類別")
                t4 = gr.Slider(1, 10, value=1, step=1, label="冊數")
                m1 = gr.Textbox(interactive=False, label="結果")
                gr.Button("入館", variant="primary").click(add_book, [t1, t2, t3, t4], m1)
                gr.Markdown("#### 辦證")
                n1 = gr.Textbox(label="姓名"); n2 = gr.Textbox(label="email")
                m2 = gr.Textbox(interactive=False, label="結果")
                gr.Button("辦證").click(register_member, [n1, n2, txt_business_date], m2)
            with gr.Column():
                gr.Markdown("#### 查書")
                k = gr.Textbox(label="書名／作者關鍵字")
                tbl_books = gr.Dataframe(value=search_books(""))
                k.change(lambda s: search_books(s), k, tbl_books)
    with gr.Tab("📊 統計報表"):
        dd_report = gr.Dropdown(list(REPORTS), value="熱門書 Top 10", label="選一張報表")
        tbl_report = gr.Dataframe(value=run_report("熱門書 Top 10"))
        dd_report.change(run_report, [dd_report, txt_business_date], tbl_report)
        txt_business_date.change(run_report, [dd_report, txt_business_date], tbl_report)
        with gr.Row():
            gr.Plot(value=trend_chart())
            gr.Plot(value=category_chart())
print("✅ 三分頁組裝完成（流通櫃台／後台管理／統計報表）；下一格 launch")

In [ ]:
# [SKIP-TEST] 開館！（Colab 內嵌；報告時 app.launch(share=True) 給全班手機操作）
app.launch(height=700)

# §9 測試總驗收（共同要求 8：≥8 assert、含 ≥2 個應該失敗）

前面每寫一個函數就測一個（散裝 assert 已超過 10 個）。這裡再做一次**總驗收**——
交專題前跑到全綠，demo 當天心臟不會痛。

In [ ]:
checks = []
def check(label, ok):
    checks.append((label, ok)); print(("✅" if ok else "❌"), label)

# --- 正常流程 ---
bid_x = lcon.execute("SELECT book_id FROM book WHERE available_copies > 0 ORDER BY book_id DESC LIMIT 1").fetchone()[0]
r0 = lcon.execute("SELECT available_copies FROM book WHERE book_id = ?", (bid_x,)).fetchone()[0]
msg = borrow_book(3, bid_x);   check("借書成功", msg.startswith("✅"))
r1 = lcon.execute("SELECT available_copies FROM book WHERE book_id = ?", (bid_x,)).fetchone()[0]
check("借書後庫存 -1", r1 == r0 - 1)
loan_x = lcon.execute("""SELECT loan_id FROM loan WHERE member_id = 3 AND book_id = ?
                         AND return_date IS NULL""", (bid_x,)).fetchone()[0]
check("還書成功", return_book(loan_x).startswith("✅"))
r2 = lcon.execute("SELECT available_copies FROM book WHERE book_id = ?", (bid_x,)).fetchone()[0]
check("還書後庫存恢復", r2 == r0)
check("辦證成功", register_member("總驗收員", "final@stat.example.edu").startswith("✅"))
check("查書有結果", len(search_books("")) > 0)

# --- 應該失敗（守門員都醒著）---
check("【應該失敗】重複 email 辦證被擋", register_member("冒名者", "final@stat.example.edu").startswith("❌"))
bid_none = lcon.execute("SELECT book_id FROM book WHERE available_copies = 0 LIMIT 1").fetchone()[0]
check("【應該失敗】0 冊可借被擋", borrow_book(1, bid_none).startswith("❌"))
check("【應該失敗】還不存在的單被擋", return_book(9_999_999).startswith("❌"))
bid_lent = lcon.execute("SELECT book_id FROM loan WHERE return_date IS NULL LIMIT 1").fetchone()[0]
check("【應該失敗】有外借的書不能刪", delete_book(bid_lent).startswith("❌"))
check("【應該失敗】有歷史借閱的書改用軟刪", "歷史借閱紀錄" in delete_book(bid_history))
check("【應該失敗】非 ISO 借書日期被擋", borrow_book(1, bid_x, "2026/11/15").startswith("❌"))

# --- 一致性稽核（快照 vs 即算，永遠的安全網）---
n_bad = lcon.execute("""SELECT COUNT(*) FROM book b
    WHERE b.available_copies <> b.total_copies -
          (SELECT COUNT(*) FROM loan l WHERE l.book_id = b.book_id AND l.return_date IS NULL)""").fetchone()[0]
check("庫存快照 = 即算（全館對帳）", n_bad == 0)

assert all(ok for _, ok in checks), "有測試沒過！"
print(f"\n🎉 總驗收 {len(checks)} 項全數通過（其中「應該失敗」{sum('應該失敗' in n for n, _ in checks)} 項）")

# §10 AI 使用說明（共同要求 9——這節示範「怎麼寫」）

- **使用工具**：Claude、Gemini。
- **我請它做什麼**（關鍵 prompt 摘錄）：
  1. 「以下是圖書館系統的需求（貼 §1 訪談稿），請做名詞動詞分析、給 3NF 的 SQLite DDL，並說明每條約束對應需求哪句話。」
  2. 「寫一個借書函數：交易保護、0 冊可借要擋、用 `?` 傳值；並給 3 個 assert（含一個應該失敗）。」
  3. 「這個 schema 撐得住『兩人同時借最後一本書』嗎？給我重現步驟。」（反向質詢）
- **它哪裡不對、我怎麼修**：
  - 初版 DDL 給了 `AUTOINCREMENT`（不必要）與 `VARCHAR(255)`（SQLite 直接用 TEXT）——依 U02 的慣例改掉；
  - 借書函數第一版是「先 SELECT 檢查再 UPDATE」——用 §5 的兩連線實驗證明會超賣，改成條件式 UPDATE；
  - 還書時它忘了喚醒預約——對照需求逐句檢查補上，並把三步包進同一交易。
- **我如何確認正確**：每個函數配 assert（§9 總驗收 13 項）；庫存用「快照 vs 即算」對帳查詢稽核；
  報表用 pandas 重算一次交叉驗證（U03 的雙引擎驗證法）。
- **心得**：AI 對「單一函數」的產出很快，但**跨步驟的一致性**（庫存同步、預約喚醒、競態）它不會主動想到——
  需求逐句檢查與「應該失敗」的測試，是人的工作。

# §11 十二分鐘 demo 腳本與備援（共同要求 10）

## 六步腳本（先寫時間，再挑畫面）

| 時間 | 要說／要操作的事 | 成功訊號 |
|---|---|---|
| 0:00–1:00 | 一句需求＋schema 決策 | 聽眾知道誰借什麼、哪條規則最重要 |
| 1:00–2:30 | DDL 約束踩點＋固定 seed 資料與對帳 | 錯資料被擋，庫存不一致為 0 |
| 2:30–4:00 | 借一本、還一本；指出同一交易的三步 | 訊息成功、庫存減一再恢復 |
| 4:00–7:00 | 先演超賣，再跑條件式 UPDATE | 壞版到 -1；好版只一人成功 |
| 7:00–10:00 | 只挑兩張報表＋索引前後證據 | 每張先說問題；計畫從 SCAN 變 SEARCH |
| 10:00–12:00 | Gradio 借→還→逾期刷新，最後總驗收 | UI 與資料同步，所有檢查全綠 |

## 三層備援

1. **A 案：live app**——報告前重跑全本，依上表逐步操作。
2. **B 案：notebook 輸出**——介面若失靈，直接展示先跑好的函數、表格、圖與 assert；講述順序不變。
3. **C 案：靜態證據**——網路或 Colab 中斷時，用事先截好的關鍵畫面／短錄影，並口頭說出每一步的預期資料變化。

## 觀摩重點

- 每條規則都有約束、交易或程式驗證負責；函數先測，UI 最後才接。
- 報表標題要寫「它回答什麼問題」，不是翻譯 SQL。
- `DEMO_TODAY` 讓資料與現場操作可重現；換日期必須使用 `YYYY-MM-DD`。
- 還書時的「已通知」**沒有替該會員保留冊數**；這是刻意簡化，正式系統應再加保留對象與期限。
- 對照自己的題目：「借書」對應「搶名額／訂位／成交」——結構同款，領域換掉。

---
## 附錄：與指派題目的關係、重跑須知

- 本示範是「**借還／流轉／狀態**」類題型的完成度標竿；
  但它的領域（圖書館）**不在指派清單內**——結構請學走，程式請自己寫（報告時任指一段要能解釋）。
- 重跑：`執行階段 → 全部執行`。資料固定 seed=42、模擬今日固定為 `2026-11-15`，跑幾次都同一份；
  `launch()` 那格在 Colab 會內嵌介面，報告時改 `share=True` 可讓同學手機操作。
- 想改造練手：換一組 seed 看報表怎麼變；把「續借規則」改成「預約者優先 3 天」；幫逾期加罰金欄。